# Ridge Regression Artwork Pricer

Regularized log-linear Ridge regression baseline for artwork valuation.

This notebook uses `../../Data/sampled_sets`:
- train: `Artwork_2026_finetune_40000_2023_2024_train_sample10000.csv`
- test: 2025 quarterly CSV files sampled to `TEST_ROWS_PER_QUARTER` rows each

The notebook trains two comparable Ridge models:
- `Ridge Regression - No Estimates`: uses artwork, artist, date, size, medium, material, auction house, location, and market features.
- `Ridge Regression - With Log Estimates`: uses the same features plus cleaned/log-scaled auction-house estimate inputs.

Both models train on `log1p(ah_price_usd)` and back-transform predictions with `expm1`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from ridge_pricer.features import (
    load_artwork_data,
    split_train_test,
    make_model_frame,
    fit_feature_schema,
    apply_feature_schema,
    print_feature_summary,
)
from ridge_pricer.modeling import (
    DEFAULT_ALPHAS,
    fit_ridge_log_price_model,
    predict_price_from_log_model,
    evaluate_price_predictions,
    evaluate_by_quarter,
    evaluate_by_price_bucket,
    get_top_coefficients,
    ridge_training_history,
)

DATA_DIR = Path("../../Data/sampled_sets_100k")
TARGET_COL = "ah_price_usd"
TRAIN_ROWS = 40_000
TEST_ROWS_PER_QUARTER = 1000
SPLIT_RANDOM_STATE = 42
SPLIT_KEY_COL = "lot_url"
CAP_OVERSIZED_REQUESTS = True
MIN_CATEGORY_COUNT = 20
VERBOSE_PREPROCESSING = True
VERBOSE_MODEL = True
ALPHAS = DEFAULT_ALPHAS
RIDGE_VALIDATION_FRACTION = 0.10  # set 0 to use RidgeCV without alpha-by-alpha diagnostics
RIDGE_LOG_PERIOD = 1
RIDGE_EVAL_METRIC = "log_RMSE"

MODEL_NO_ESTIMATES = "Ridge Regression - No Estimates"
MODEL_WITH_ESTIMATES = "Ridge Regression - With Log Estimates"

## 1. Load Sampled Data

In [2]:
raw_df = load_artwork_data(DATA_DIR, verbose=VERBOSE_PREPROCESSING)
print(raw_df.shape)
raw_df[["source_file", "source_quarter", "sale_date", "ah_price_usd", "lot_url"]].head()

[ridge features] Loading artwork files from /srv/notebooks/[6.0] ML Pricers/Data/sampled_sets_100k
[ridge features]   loaded Artwork_2026_finetune_100000_2022_2024_train_all_eligible.csv: 38,261 rows, 70 columns
[ridge features]   loaded Artwork_2026_finetune_100000_q2025Q1_sample1000.csv: 1,000 rows, 70 columns
[ridge features]   loaded Artwork_2026_finetune_100000_q2025Q2_sample1000.csv: 1,000 rows, 70 columns
[ridge features]   loaded Artwork_2026_finetune_100000_q2025Q3_sample1000.csv: 1,000 rows, 70 columns
[ridge features]   loaded Artwork_2026_finetune_100000_q2025Q4_sample1000.csv: 1,000 rows, 70 columns
[ridge features] Loaded combined dataframe: 42,261 rows, 70 columns
(42261, 70)


,source_file,source_quarter,sale_date,ah_price_usd,lot_url
0,Artwork_2026_finetune_100000_2022_2024_train_a...,None,2022-07-01,31771.00,https://www.christies.com/en/lot/lot-6380380?l...
1,Artwork_2026_finetune_100000_2022_2024_train_a...,None,2024-11-20,63000.00,https://www.christies.com/en/lot/lot-6508480?l...
2,Artwork_2026_finetune_100000_2022_2024_train_a...,None,2023-11-23,746.65,https://www.sothebys.com/en/buy/auction/2023/m...
3,Artwork_2026_finetune_100000_2022_2024_train_a...,None,2022-09-23,11299.27,https://www.sothebys.com/en/buy/auction/2022/m...
4,Artwork_2026_finetune_100000_2022_2024_train_a...,None,2024-11-07,1852.50,https://hindmanauctions.com/auctions/2084-19th...


## 2. Train/Test Split

Uses `SPLIT_KEY_COL = "lot_url"` for duplicate/leakage checks. The train row count and per-quarter test row count are notebook variables. If `CAP_OVERSIZED_REQUESTS = True`, oversized requests are capped to the maximum rows available after filtering.

In [3]:
train_raw, test_raw = split_train_test(
    raw_df,
    train_rows=TRAIN_ROWS,
    test_rows_per_quarter=TEST_ROWS_PER_QUARTER,
    dedupe_key_col=SPLIT_KEY_COL,
    cap_oversized_requests=CAP_OVERSIZED_REQUESTS,
    random_state=SPLIT_RANDOM_STATE,
    verbose=VERBOSE_PREPROCESSING,
)
print("train raw:", train_raw.shape)
print("test raw:", test_raw.shape)
print("test quarters:")
print(test_raw["source_quarter"].value_counts().sort_index())
print("train/test lot_url overlap:", len(set(train_raw["lot_url"].dropna()) & set(test_raw["lot_url"].dropna())))

[ridge features] Initial split: train=38,261, explicit test=4,000
[ridge features] Requested 40,000 train rows, only 38,261 available; using all available train rows.
[ridge features] Using train rows: 38,261
[ridge features] Train contains 4,001 duplicate lot_url rows; keeping them to preserve exact train size.
[ridge features] De-duplicated test lot_url: 4,000->4,000
[ridge features] Removed train/test overlapping lot_url values from test: 4,000->4,000
[ridge features] Requested test rows per quarter: 1,000
[ridge features] Actual test rows per quarter: {'2025Q1': 1000, '2025Q2': 1000, '2025Q3': 1000, '2025Q4': 1000}
[ridge features] Final split: train=38,261, test=4,000
train raw: (38261, 70)
test raw: (4000, 70)
test quarters:
source_quarter
2025Q1    1000
2025Q2    1000
2025Q3    1000
2025Q4    1000
Name: count, dtype: int64
train/test lot_url overlap: 0


## 3. Feature Engineering: No Estimates

This is the baseline Ridge feature set. It follows the agreed forced inputs, with `sale_date` and `year_of_creation` moved to numeric features. Auction estimate fields are retained for reporting but excluded from model training.

In [4]:
train_frame, feature_cols, categorical_cols, numeric_cols = make_model_frame(
    train_raw,
    target_col=TARGET_COL,
    include_estimates=False,
    verbose=VERBOSE_PREPROCESSING,
)
test_frame, _, _, _ = make_model_frame(
    test_raw,
    target_col=TARGET_COL,
    include_estimates=False,
    verbose=VERBOSE_PREPROCESSING,
)

schema = fit_feature_schema(
    train_frame,
    feature_cols,
    categorical_cols,
    min_category_count=MIN_CATEGORY_COUNT,
    verbose=VERBOSE_PREPROCESSING,
)

X_train = apply_feature_schema(train_frame, feature_cols, categorical_cols, schema, verbose=VERBOSE_PREPROCESSING)
X_test = apply_feature_schema(test_frame, feature_cols, categorical_cols, schema, verbose=VERBOSE_PREPROCESSING)
y_train_log = train_frame["log_target_price_usd"]

print("X_train no estimates:", X_train.shape)
print("X_test no estimates:", X_test.shape)
feature_report = print_feature_summary(
    train_frame,
    feature_cols,
    categorical_cols,
    numeric_cols,
    schema=schema,
)
feature_report

[ridge features] Building Ridge model frame from 38,261 rows using target_col='ah_price_usd', include_estimates=False
[ridge features] Filtered invalid/non-positive targets: 38,261->38,261
[ridge features] Engineered 24 features (10 categorical, 14 numeric).
[ridge features]   auction estimate fields retained for reporting but excluded from model features
[ridge features] Building Ridge model frame from 4,000 rows using target_col='ah_price_usd', include_estimates=False
[ridge features] Filtered invalid/non-positive targets: 4,000->4,000
[ridge features] Engineered 24 features (10 categorical, 14 numeric).
[ridge features]   auction estimate fields retained for reporting but excluded from model features
[ridge features] Fitting Ridge feature schema with min_category_count=20
[ridge features]   categorical auction_house: 7 kept levels from 5 observed
[ridge features]   categorical location: 13 kept levels from 12 observed
[ridge features]   categorical material_clean: 137 kept levels fr

,feature,role,transformation,missing_rate,unique_count_before_grouping,train_schema_levels,example_values
0,auction_house,categorical,Cleaned categorical from raw auction_house; ra...,0.000000,5,7.0,"Christies, Sotheby's, Hindman"
1,location,categorical,Cleaned categorical from raw location; rare le...,0.000000,12,13.0,"London, New York, Hong Kong"
2,material_clean,categorical,Lexically cleaned raw material text; rare clea...,0.000000,10658,137.0,"coloured pencil graphite paper, cast stone, wa..."
3,artwork_title_clean,categorical,Cleaned categorical from artwork_title; rare t...,0.000000,29326,20.0,"""Stamped Indelibly"" mouth, Liseur dans un faut..."
4,medium,categorical,Cleaned categorical from raw medium; rare leve...,0.000000,12,12.0,"drawing, sculpture, painting"
5,signed,categorical,Cleaned categorical yes/no value from signed.,0.000000,2,4.0,"yes, no"
6,titled,categorical,Cleaned categorical yes/no value from titled.,0.000000,2,4.0,"yes, no"
7,dated,categorical,Cleaned categorical yes/no value from dated.,0.000000,2,4.0,"yes, no"
8,artist_name_grouped,categorical,Cleaned categorical from artist_name; rare art...,0.000000,9983,284.0,"Tom Wesselmann, Jacques Lipchitz, Dong Kingman"
9,edition_clean,categorical,Cleaned categorical from edition; rare levels ...,0.000000,1,2.0,__missing__


## 4. Feature Engineering: With Estimates

This version uses the same rows and base features, then adds cleaned/log-scaled auction estimate inputs:
- `log_auction_low_estimate_usd`
- `log_auction_high_estimate_usd`
- `log_auction_estimate_midpoint_usd`
- `auction_estimate_range_ratio`

Raw estimate USD values are retained for reporting/export, but they are not used as Ridge training features. Zero or negative estimates are treated as missing before log transformation.

In [5]:
train_frame_est, feature_cols_est, categorical_cols_est, numeric_cols_est = make_model_frame(
    train_raw,
    target_col=TARGET_COL,
    include_estimates=True,
    verbose=VERBOSE_PREPROCESSING,
)
test_frame_est, _, _, _ = make_model_frame(
    test_raw,
    target_col=TARGET_COL,
    include_estimates=True,
    verbose=VERBOSE_PREPROCESSING,
)

schema_est = fit_feature_schema(
    train_frame_est,
    feature_cols_est,
    categorical_cols_est,
    min_category_count=MIN_CATEGORY_COUNT,
    verbose=VERBOSE_PREPROCESSING,
)

X_train_est = apply_feature_schema(train_frame_est, feature_cols_est, categorical_cols_est, schema_est, verbose=VERBOSE_PREPROCESSING)
X_test_est = apply_feature_schema(test_frame_est, feature_cols_est, categorical_cols_est, schema_est, verbose=VERBOSE_PREPROCESSING)

print("X_train with estimates:", X_train_est.shape)
print("X_test with estimates:", X_test_est.shape)
feature_report_est = print_feature_summary(
    train_frame_est,
    feature_cols_est,
    categorical_cols_est,
    numeric_cols_est,
    schema=schema_est,
)
feature_report_est

[ridge features] Building Ridge model frame from 38,261 rows using target_col='ah_price_usd', include_estimates=True
[ridge features] Filtered invalid/non-positive targets: 38,261->38,261
[ridge features] Engineered 28 features (10 categorical, 18 numeric).
[ridge features]   included auction estimate numeric features: ['log_auction_low_estimate_usd', 'log_auction_high_estimate_usd', 'log_auction_estimate_midpoint_usd', 'auction_estimate_range_ratio']
[ridge features] Building Ridge model frame from 4,000 rows using target_col='ah_price_usd', include_estimates=True
[ridge features] Filtered invalid/non-positive targets: 4,000->4,000
[ridge features] Engineered 28 features (10 categorical, 18 numeric).
[ridge features]   included auction estimate numeric features: ['log_auction_low_estimate_usd', 'log_auction_high_estimate_usd', 'log_auction_estimate_midpoint_usd', 'auction_estimate_range_ratio']
[ridge features] Fitting Ridge feature schema with min_category_count=20
[ridge features]  

,feature,role,transformation,missing_rate,unique_count_before_grouping,train_schema_levels,example_values
0,auction_house,categorical,Cleaned categorical from raw auction_house; ra...,0.000000,5,7.0,"Christies, Sotheby's, Hindman"
1,location,categorical,Cleaned categorical from raw location; rare le...,0.000000,12,13.0,"London, New York, Hong Kong"
2,material_clean,categorical,Lexically cleaned raw material text; rare clea...,0.000000,10658,137.0,"coloured pencil graphite paper, cast stone, wa..."
3,artwork_title_clean,categorical,Cleaned categorical from artwork_title; rare t...,0.000000,29326,20.0,"""Stamped Indelibly"" mouth, Liseur dans un faut..."
4,medium,categorical,Cleaned categorical from raw medium; rare leve...,0.000000,12,12.0,"drawing, sculpture, painting"
5,signed,categorical,Cleaned categorical yes/no value from signed.,0.000000,2,4.0,"yes, no"
6,titled,categorical,Cleaned categorical yes/no value from titled.,0.000000,2,4.0,"yes, no"
7,dated,categorical,Cleaned categorical yes/no value from dated.,0.000000,2,4.0,"yes, no"
8,artist_name_grouped,categorical,Cleaned categorical from artist_name; rare art...,0.000000,9983,284.0,"Tom Wesselmann, Jacques Lipchitz, Dong Kingman"
9,edition_clean,categorical,Cleaned categorical from edition; rare levels ...,0.000000,1,2.0,__missing__


## 5. Train Ridge Models With Alpha Diagnostics

In [6]:
ridge_model = fit_ridge_log_price_model(
    X_train,
    y_train_log,
    categorical_cols=categorical_cols,
    numeric_cols=numeric_cols,
    alphas=ALPHAS,
    validation_fraction=RIDGE_VALIDATION_FRACTION,
    log_period=RIDGE_LOG_PERIOD,
    eval_metric=RIDGE_EVAL_METRIC,
    random_state=SPLIT_RANDOM_STATE,
    verbose=VERBOSE_MODEL,
)
test_pred = predict_price_from_log_model(ridge_model, X_test)

ridge_model_est = fit_ridge_log_price_model(
    X_train_est,
    y_train_log,
    categorical_cols=categorical_cols_est,
    numeric_cols=numeric_cols_est,
    alphas=ALPHAS,
    validation_fraction=RIDGE_VALIDATION_FRACTION,
    log_period=RIDGE_LOG_PERIOD,
    eval_metric=RIDGE_EVAL_METRIC,
    random_state=SPLIT_RANDOM_STATE,
    verbose=VERBOSE_MODEL,
)
test_pred_est = predict_price_from_log_model(ridge_model_est, X_test_est)

print("Selected alpha no estimates:", ridge_model.selected_alpha_)
print("Selected alpha with estimates:", ridge_model_est.selected_alpha_)

training_history = ridge_training_history(ridge_model)
training_history.insert(0, "model", MODEL_NO_ESTIMATES)

training_history_est = ridge_training_history(ridge_model_est)
training_history_est.insert(0, "model", MODEL_WITH_ESTIMATES)

training_history_comparison = pd.concat([training_history, training_history_est], ignore_index=True)
training_history_comparison

[ridge model] Training Ridge on 38,261 rows and 24 features
[ridge model] Alpha grid: [0.001, 0.0029286445646252374, 0.008576958985908946, 0.025118864315095794, 0.07356422544596414, 0.21544346900318845, 0.630957344480193, 1.8478497974222907, 5.411695265464638, 15.848931924611142, 46.41588833612782, 135.93563908785268, 398.1071705534969, 1165.9144011798312, 3414.5488738336007, 10000.0]
[ridge model] Running alpha diagnostics: train_fit=34,434, validation=3,827, eval_metric=log_RMSE, log_period=1
[ridge model] [01/16] alpha=0.001 log_RMSE=1.49088 log_MAE=1.14543 RMSE=1536919.95 MAE=211310.62
[ridge model] [02/16] alpha=0.00292864 log_RMSE=1.49089 log_MAE=1.14540 RMSE=1536983.92 MAE=211321.46
[ridge model] [03/16] alpha=0.00857696 log_RMSE=1.49087 log_MAE=1.14545 RMSE=1537016.94 MAE=211342.15
[ridge model] [04/16] alpha=0.0251189 log_RMSE=1.49086 log_MAE=1.14545 RMSE=1537100.91 MAE=211350.18
[ridge model] [05/16] alpha=0.0735642 log_RMSE=1.49086 log_MAE=1.14543 RMSE=1537307.93 MAE=211348.

,model,iteration,alpha,log_MAE,log_RMSE,log_R2,MAE,RMSE,MAPE,R2
0,Ridge Regression - No Estimates,1,0.001000,1.145430,1.490882,0.518014,211310.621610,1.536920e+06,2.996128,0.139667
1,Ridge Regression - No Estimates,2,0.002929,1.145401,1.490888,0.518010,211321.464273,1.536984e+06,2.996117,0.139595
2,Ridge Regression - No Estimates,3,0.008577,1.145450,1.490866,0.518025,211342.151981,1.537017e+06,2.996242,0.139558
3,Ridge Regression - No Estimates,4,0.025119,1.145452,1.490862,0.518027,211350.183322,1.537101e+06,2.996223,0.139464
4,Ridge Regression - No Estimates,5,0.073564,1.145432,1.490856,0.518031,211348.379316,1.537308e+06,2.995978,0.139232
5,Ridge Regression - No Estimates,6,0.215443,1.145454,1.490806,0.518064,211448.248555,1.538111e+06,2.995915,0.138333
6,Ridge Regression - No Estimates,7,0.630957,1.145566,1.490829,0.518048,211729.187322,1.540308e+06,2.995436,0.135869
7,Ridge Regression - No Estimates,8,1.847850,1.145900,1.490964,0.517961,212424.970582,1.546118e+06,2.995794,0.129338
8,Ridge Regression - No Estimates,9,5.411695,1.148167,1.493110,0.516573,214142.679403,1.560397e+06,3.002235,0.113182
9,Ridge Regression - No Estimates,10,15.848932,1.159093,1.503821,0.509612,217827.493074,1.585349e+06,3.037535,0.084593


## 6. Overall Comparison Metrics

Only the approved metrics are reported: `MAE`, `RMSE`, `MAPE`, and `R2`.

In [7]:
overall_metrics = pd.DataFrame([
    evaluate_price_predictions(test_frame["target_price_usd"], test_pred, label=MODEL_NO_ESTIMATES),
    evaluate_price_predictions(test_frame_est["target_price_usd"], test_pred_est, label=MODEL_WITH_ESTIMATES),
])
overall_metrics

,model,n,MAE,RMSE,MAPE,R2
0,Ridge Regression - No Estimates,4000,116385.254497,767508.275594,3.067082,-0.172310
1,Ridge Regression - With Log Estimates,4000,38654.834341,225154.969957,0.762924,0.899112


## 7. Quarterly Comparison Metrics

In [8]:
quarter_metrics = pd.concat(
    [
        evaluate_by_quarter(test_frame, test_pred, label=MODEL_NO_ESTIMATES),
        evaluate_by_quarter(test_frame_est, test_pred_est, label=MODEL_WITH_ESTIMATES),
    ],
    ignore_index=True,
).sort_values(["quarter", "model"]).reset_index(drop=True)
quarter_metrics

,model,n,MAE,RMSE,MAPE,R2,quarter
0,Ridge Regression - No Estimates,1000,66497.752206,3.792236e+05,2.579188,0.152989,2025Q1
1,Ridge Regression - With Log Estimates,1000,28944.342731,1.805496e+05,0.779289,0.808004,2025Q1
2,Ridge Regression - No Estimates,1000,195835.365147,1.168088e+06,2.124852,-0.372962,2025Q2
3,Ridge Regression - With Log Estimates,1000,48357.572981,2.032525e+05,0.839344,0.958430,2025Q2
4,Ridge Regression - No Estimates,1000,38690.533265,2.309539e+05,5.006013,-0.643464,2025Q3
5,Ridge Regression - With Log Estimates,1000,13465.741131,6.128250e+04,0.834175,0.884287,2025Q3
6,Ridge Regression - No Estimates,1000,164517.367368,8.914575e+05,2.558273,0.002506,2025Q4
7,Ridge Regression - With Log Estimates,1000,63851.680522,3.537143e+05,0.598889,0.842959,2025Q4


## 8. Metrics By Price Range

In [9]:
price_bucket_metrics = pd.concat(
    [
        evaluate_by_price_bucket(test_frame, test_pred, label=MODEL_NO_ESTIMATES),
        evaluate_by_price_bucket(test_frame_est, test_pred_est, label=MODEL_WITH_ESTIMATES),
    ],
    ignore_index=True,
).sort_values(["price_bucket", "model"]).reset_index(drop=True)
price_bucket_metrics

,model,n,MAE,RMSE,MAPE,R2,price_bucket
0,Ridge Regression - No Estimates,302,2.306323e+05,1.145141e+06,0.910023,-116.048957,$100k-$500k
1,Ridge Regression - With Log Estimates,302,8.075869e+04,1.071207e+05,0.382613,-0.024227,$100k-$500k
2,Ridge Regression - No Estimates,1256,8.098542e+03,4.254179e+04,3.713324,-1446.578290,$1k-$5k
3,Ridge Regression - With Log Estimates,1256,2.034229e+03,8.466597e+03,0.841233,-56.336120,$1k-$5k
4,Ridge Regression - No Estimates,633,4.631752e+04,2.035392e+05,0.979467,-101.515020,$25k-$100k
5,Ridge Regression - With Log Estimates,633,1.877991e+04,2.536197e+04,0.391709,-0.591686,$25k-$100k
6,Ridge Regression - No Estimates,163,2.071106e+06,3.441755e+06,0.852813,-0.603727,$500k+
7,Ridge Regression - With Log Estimates,163,6.614840e+05,1.103680e+06,0.329832,0.835086,$500k+
8,Ridge Regression - No Estimates,1252,1.327156e+04,2.445762e+04,1.234990,-19.881385,$5k-$25k
9,Ridge Regression - With Log Estimates,1252,6.066425e+03,1.448980e+04,0.543898,-6.329190,$5k-$25k


## 9. Coefficient Inspection

Ridge is interpretable after preprocessing. Coefficients are on the log-price scale.

In [10]:
top_coefficients = get_top_coefficients(ridge_model, top_n=40)
top_coefficients.insert(0, "model", MODEL_NO_ESTIMATES)

top_coefficients_est = get_top_coefficients(ridge_model_est, top_n=40)
top_coefficients_est.insert(0, "model", MODEL_WITH_ESTIMATES)

coefficient_comparison = pd.concat([top_coefficients, top_coefficients_est], ignore_index=True)
coefficient_comparison

,model,feature,coefficient,abs_coefficient
0,Ridge Regression - No Estimates,categorical__artist_name_grouped_Claude Monet,3.614077,3.614077
1,Ridge Regression - No Estimates,categorical__artist_name_grouped_Frederick Conway,-3.541657,3.541657
2,Ridge Regression - No Estimates,categorical__artist_name_grouped_Jean-michel B...,3.344927,3.344927
3,Ridge Regression - No Estimates,categorical__artist_name_grouped_Jean-Michel B...,3.267181,3.267181
4,Ridge Regression - No Estimates,categorical__artist_name_grouped_Artist Unknown,-3.197591,3.197591
...,...,...,...,...
75,Ridge Regression - With Log Estimates,categorical__artist_name_grouped_Mark Grotjahn,-0.277301,0.277301
76,Ridge Regression - With Log Estimates,categorical__artist_name_grouped_Zhang Xiaogang,-0.275713,0.275713
77,Ridge Regression - With Log Estimates,categorical__artist_name_grouped_Caroline Walker,0.275213,0.275213
78,Ridge Regression - With Log Estimates,categorical__artist_name_grouped_After Pablo P...,0.275211,0.275211


## 10. Export Results

In [11]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

scored_test = test_frame[[
    "ID",
    "lot_url",
    "source_quarter",
    "sale_date",
    "target_price_usd",
    "auction_low_estimate_usd",
    "auction_high_estimate_usd",
    "auction_estimate_midpoint_usd",
    "auction_estimate_range_usd",
    "log_auction_low_estimate_usd",
    "log_auction_high_estimate_usd",
    "log_auction_estimate_midpoint_usd",
    "auction_estimate_range_ratio",
]].copy()
scored_test["ridge_pred_no_estimates_usd"] = test_pred
scored_test["ridge_error_no_estimates_usd"] = scored_test["ridge_pred_no_estimates_usd"] - scored_test["target_price_usd"]
scored_test["ridge_pred_with_estimates_usd"] = test_pred_est
scored_test["ridge_error_with_estimates_usd"] = scored_test["ridge_pred_with_estimates_usd"] - scored_test["target_price_usd"]

scored_test.to_csv(output_dir / "ridge_test_predictions.csv", index=False)
overall_metrics.to_csv(output_dir / "ridge_overall_metrics.csv", index=False)
quarter_metrics.to_csv(output_dir / "ridge_quarter_metrics.csv", index=False)
price_bucket_metrics.to_csv(output_dir / "ridge_price_bucket_metrics.csv", index=False)
coefficient_comparison.to_csv(output_dir / "ridge_top_coefficients.csv", index=False)
training_history_comparison.to_csv(output_dir / "ridge_training_history.csv", index=False)
feature_report.to_csv(output_dir / "ridge_feature_report_no_estimates.csv", index=False)
feature_report_est.to_csv(output_dir / "ridge_feature_report_with_estimates.csv", index=False)

print("Wrote outputs to", output_dir.resolve())

Wrote outputs to /srv/notebooks/[6.0] ML Pricers/Linear_Regression/Ridge_Regression/outputs
